In [37]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import importlib

# Import our custom modules
from parser import HazelLogParser
from path_extractor import analyze_induction_exploration_paths

# Reload path_extractor module if it's been modified
if 'path_extractor' in sys.modules:
    importlib.reload(sys.modules['path_extractor'])
    from path_extractor import analyze_induction_exploration_paths

# Set up plotting
plt.style.use('default')
sns.set_palette('husl')
%matplotlib inline

## Load Data

In [41]:
# Initialize parser
parser = HazelLogParser()
# Load log file
log_file = "a2_logs/submission_349620284.json"  # Change this to your log file
df = parser.parse_file(log_file)

## Exploration Paths After AddInduction

This section analyzes what happens after students add induction - showing exploration paths and how users backtrack using RemoveStep.


In [42]:
# Analyze exploration paths after AddInduction actions
# Try WITHOUT filtering first to see if RemoveStep actions exist
print("=== ANALYZING WITHOUT FILTERING (see all actions) ===")
exploration_analysis_no_filter = analyze_induction_exploration_paths(df, max_steps_after_add=None, ignore_focus_actions=False)
stats_no_filter = exploration_analysis_no_filter['statistics']
print(f"Total AddInduction events: {stats_no_filter['total_add_induction']}")
print(f"Removed: {stats_no_filter['removed_count']}")
if stats_no_filter.get('average_total_steps'):
    print(f"Average total steps after AddInduction: {stats_no_filter['average_total_steps']:.1f}")

print("\n=== ANALYZING WITH FILTERING (ignore focus actions) ===")
exploration_analysis = analyze_induction_exploration_paths(df, max_steps_after_add=None, ignore_focus_actions=True)

# View summary statistics
print("\n=== EXPLORATION PATH ANALYSIS (FILTERED) ===")
stats = exploration_analysis['statistics']
print(f"\nSummary Statistics:")
print(f"Total AddInduction events: {stats['total_add_induction']}")
print(f"Removed: {stats['removed_count']}")

# Use the unfiltered analysis for better results if filtering removes RemoveSteps
if stats_no_filter['removed_count'] > stats['removed_count']:
    print("\n⚠️  NOTE: Filtering removed some RemoveStep actions. Using unfiltered analysis.")
    exploration_analysis = exploration_analysis_no_filter
    stats = stats_no_filter


=== ANALYZING WITHOUT FILTERING (see all actions) ===
Total AddInduction events: 7
Removed: 6
Average total steps after AddInduction: 168.7

=== ANALYZING WITH FILTERING (ignore focus actions) ===
Ignored 4843 non-meaningful actions (4843/8466 = 57.2%)

=== EXPLORATION PATH ANALYSIS (FILTERED) ===

Summary Statistics:
Total AddInduction events: 7
Removed: 6


In [43]:
# Display example exploration paths
exploration_paths = exploration_analysis['exploration_paths']

print("=== EXAMPLE EXPLORATION PATHS ===")
print("\nFirst 5 AddInduction events and their exploration paths:")

for i, path in enumerate(exploration_paths[:5]):
    print(f"\n--- Exploration Path {i+1} ---")
    print(f"AddInduction at index: {path['add_induction']['action_index']}")
    print(f"Total steps after AddInduction: {path['total_steps_after']}")
    print(f"Was removed: {path['was_removed']}")
    print(f"Number of RemoveSteps: {path['removestep_count']}")
    
    if path['removestep_count'] > 0:
        print(f"First removal at step: {path['first_removal_step']}")
        print(f"RemoveStep positions: {[r['step_number'] for r in path['removesteps']]}")
    
    print(f"\nAction Categories:")
    for category, count in path['action_categories'].items():
        if count > 0:
            print(f"  {category}: {count}")
    
    print(f"\nFirst 10 actions after AddInduction:")
    for action in path['actions_before_removal'][:10]:
        action_str = str(action['action'])
        # Truncate long actions
        if len(action_str) > 80:
            action_str = action_str[:80] + "..."
        print(f"  Step {action['step_number']}: {action_str}")


=== EXAMPLE EXPLORATION PATHS ===

First 5 AddInduction events and their exploration paths:

--- Exploration Path 1 ---
AddInduction at index: 3347
Total steps after AddInduction: 1
Was removed: True
Number of RemoveSteps: 1
First removal at step: 3349
RemoveStep positions: [3349]

Action Categories:
  removesteps: 1

First 10 actions after AddInduction:

--- Exploration Path 2 ---
AddInduction at index: 3349
Total steps after AddInduction: 131
Was removed: True
Number of RemoveSteps: 3
First removal at step: 3382
RemoveStep positions: [3382, 3458, 3481]

Action Categories:
  case_modifications: 124
  removesteps: 3
  other: 4

First 10 actions after AddInduction:
  Step 3351: Editors(Exercises(TheoremExercise(Theorem(ResultAction(TheoremsAction(TheoremUpd...
  Step 3352: Editors(Exercises(TheoremExercise(Theorem(ResultAction(TheoremsAction(TheoremUpd...
  Step 3353: Editors(Exercises(TheoremExercise(Theorem(ResultAction(TheoremsAction(TheoremUpd...
  Step 3354: Editors(Exercises(Theor

In [44]:
# Find paths with multiple RemoveSteps (backtracking behavior)
backtracking_sequences = exploration_analysis['backtracking_patterns']['removestep_sequences']

if backtracking_sequences:
    print("=== PATHS WITH MULTIPLE REMOVESTEPS (BACKTRACKING) ===")
    print(f"Found {len(backtracking_sequences)} paths with multiple RemoveSteps")
    
    for i, seq in enumerate(backtracking_sequences[:5]):
        print(f"\nBacktracking Sequence {i+1}:")
        print(f"  AddInduction at index: {seq['add_induction_index']}")
        print(f"  Total RemoveSteps: {seq['removestep_count']}")
        print(f"  RemoveStep at steps: {seq['removestep_steps']}")
        
        # Show the full exploration path
        path_idx = None
        for j, path in enumerate(exploration_paths):
            if path['add_induction']['action_index'] == seq['add_induction_index']:
                path_idx = j
                break
        
        if path_idx is not None:
            path = exploration_paths[path_idx]
            print(f"  Total steps: {path['total_steps_after']}")
            print(f"  Action categories: {path['action_categories']}")
else:
    print("No paths found with multiple RemoveSteps")


=== PATHS WITH MULTIPLE REMOVESTEPS (BACKTRACKING) ===
Found 3 paths with multiple RemoveSteps

Backtracking Sequence 1:
  AddInduction at index: 3349
  Total RemoveSteps: 3
  RemoveStep at steps: [3382, 3458, 3481]
  Total steps: 131
  Action categories: {'case_modifications': 124, 'axiom_additions': 0, 'step_forwards': 0, 'other_step_creations': 0, 'removesteps': 3, 'other': 4}

Backtracking Sequence 2:
  AddInduction at index: 3481
  Total RemoveSteps: 2
  RemoveStep at steps: [3485, 3487]
  Total steps: 40
  Action categories: {'case_modifications': 38, 'axiom_additions': 0, 'step_forwards': 0, 'other_step_creations': 0, 'removesteps': 2, 'other': 0}

Backtracking Sequence 3:
  AddInduction at index: 3526
  Total RemoveSteps: 18
  RemoveStep at steps: [3539, 3540, 3541, 3542, 3543, 3544, 3550, 3573, 3574, 3579, 3580, 3581, 3582, 3586, 3589, 3590, 3591, 3592]
  Total steps: 75
  Action categories: {'case_modifications': 55, 'axiom_additions': 0, 'step_forwards': 0, 'other_step_creat

In [46]:
# Analyze removed paths
removed_paths = [p for p in exploration_paths if p['was_removed']]
[p['total_steps_after'] for p in removed_paths]

[1, 131, 40, 1, 1, 75]

## Aggregate Analysis Across All Log Files

In [19]:
import os
import glob
from path_extractor import analyze_induction_exploration_paths

# Find all JSON files in a2_logs
log_files = sorted(glob.glob("a2_logs/submission_*.json"))
print(f"Found {len(log_files)} log files")
print(f"\nFirst 5 files: {[os.path.basename(f) for f in log_files[:5]]}")

Found 25 log files

First 5 files: ['submission_348406434.json', 'submission_348413518.json', 'submission_348627900.json', 'submission_348781861.json', 'submission_349144175.json']


In [20]:
# Process all files and collect results
parser = HazelLogParser()
all_results = []
failed_files = []

for log_file in log_files:
    try:
        print(f"Processing {os.path.basename(log_file)}...", end=" ")
        df = parser.parse_file(log_file)
        
        # Run analysis without filtering to get complete picture
        result = analyze_induction_exploration_paths(df, max_steps_after_add=None, ignore_focus_actions=False)
        
        # Add file identifier
        result['filename'] = os.path.basename(log_file)
        result['filepath'] = log_file
        result['total_actions'] = len(df)
        
        all_results.append(result)
        print(f"✓ ({result['statistics']['total_add_induction']} AddInduction events)")
    except Exception as e:
        print(f"✗ Error: {e}")
        failed_files.append((log_file, str(e)))

print(f"\n\nProcessed {len(all_results)} files successfully")
if failed_files:
    print(f"Failed to process {len(failed_files)} files")

Processing submission_348406434.json... ✓ (8 AddInduction events)
Processing submission_348413518.json... ✓ (4 AddInduction events)
Processing submission_348627900.json... ✓ (21 AddInduction events)
Processing submission_348781861.json... ✓ (44 AddInduction events)
Processing submission_349144175.json... ✓ (8 AddInduction events)
Processing submission_349265175.json... ✓ (10 AddInduction events)
Processing submission_349351711.json... ✓ (11 AddInduction events)
Processing submission_349441241.json... ✓ (16 AddInduction events)
Processing submission_349549369.json... ✓ (12 AddInduction events)
Processing submission_349620284.json... ✓ (7 AddInduction events)
Processing submission_349639227.json... ✓ (19 AddInduction events)
Processing submission_349810036.json... ✓ (13 AddInduction events)
Processing submission_349835873.json... ✓ (38 AddInduction events)
Processing submission_349856151.json... ✓ (8 AddInduction events)
Processing submission_349944161.json... ✓ (14 AddInduction events)


In [21]:
# Aggregate statistics across all files
aggregated_stats = {
    'total_files': len(all_results),
    'total_add_induction_events': sum(r['statistics']['total_add_induction'] for r in all_results),
    'total_removed': sum(r['statistics']['removed_count'] for r in all_results),
    'files_with_removals': sum(1 for r in all_results if r['statistics']['removed_count'] > 0),
}

if aggregated_stats['total_add_induction_events'] > 0:
    aggregated_stats['overall_removal_rate'] = aggregated_stats['total_removed'] / aggregated_stats['total_add_induction_events']
else:
    aggregated_stats['overall_removal_rate'] = 0

print("=== AGGREGATED STATISTICS ACROSS ALL FILES ===")
print(f"\nTotal files processed: {aggregated_stats['total_files']}")
print(f"Total AddInduction events: {aggregated_stats['total_add_induction_events']}")
print(f"Total removed: {aggregated_stats['total_removed']}")
print(f"\nOverall removal rate: {aggregated_stats['overall_removal_rate']:.2%}")
print(f"\nFiles with removals: {aggregated_stats['files_with_removals']}/{aggregated_stats['total_files']}")

=== AGGREGATED STATISTICS ACROSS ALL FILES ===

Total files processed: 25
Total AddInduction events: 361
Total removed: 236

Overall removal rate: 65.37%

Files with removals: 24/25


In [22]:
# Create DataFrame for detailed analysis
file_stats = []

for result in all_results:
    stats = result['statistics']
    backtrack = result['backtracking_patterns']
    
    file_stats.append({
        'filename': result['filename'],
        'total_actions': result['total_actions'],
        'add_induction_count': stats['total_add_induction'],
        'removed_count': stats['removed_count'],
        'removal_rate': stats['removed_count'] / stats['total_add_induction'] if stats['total_add_induction'] > 0 else 0,
        'single_removestep': backtrack['single_removestep'],
        'multiple_removesteps': backtrack['multiple_removesteps'],
        'avg_steps_before_removal': stats.get('average_steps_before_removal'),
        'avg_total_steps': stats.get('average_total_steps', 0),
    })

stats_df = pd.DataFrame(file_stats)
# Summary statistics
print("\n=== SUMMARY STATISTICS ===")
print(f"Mean removal rate: {stats_df['removal_rate'].mean():.2%}")
print(f"Median removal rate: {stats_df['removal_rate'].median():.2%}")
print(f"Mean AddInduction per file: {stats_df['add_induction_count'].mean():.1f}")
print(f"Mean removed per file: {stats_df['removed_count'].mean():.1f}")

stats_df


=== SUMMARY STATISTICS ===
Mean removal rate: 72.89%
Median removal rate: 85.71%
Mean AddInduction per file: 14.4
Mean removed per file: 9.4


,filename,total_actions,add_induction_count,removed_count,removal_rate,single_removestep,multiple_removesteps,avg_steps_before_removal,avg_total_steps
0,submission_348406434.json,20929,8,4,0.500000,0,4,19699.000000,361.625000
1,submission_348413518.json,5750,4,2,0.500000,0,2,5213.500000,209.750000
2,submission_348627900.json,10393,21,18,0.857143,8,10,6646.222222,231.619048
3,submission_348781861.json,6280,44,12,0.272727,0,12,3561.333333,120.818182
4,submission_349144175.json,2645,8,8,1.000000,4,4,740.375000,281.875000
5,submission_349265175.json,11073,10,9,0.900000,3,6,9875.777778,205.200000
6,submission_349351711.json,12778,11,7,0.636364,3,4,12103.857143,86.818182
7,submission_349441241.json,3981,16,16,1.000000,2,14,952.312500,246.562500
8,submission_349549369.json,10847,12,11,0.916667,5,6,6684.727273,401.083333
9,submission_349620284.json,8466,7,6,0.857143,3,3,7751.000000,168.714286
